# Imports

In [1]:
from scipy.io import loadmat
from scipy.stats import mode
import numpy as np
import matplotlib.pyplot as plt
import os
from joblib import Parallel, delayed
import mne
from scipy.signal import butter, filtfilt, find_peaks, decimate, savgol_filter
from collections import defaultdict
from scipy.signal import hilbert
import pandas as pd
import h5py

# Functions

In [2]:
def wei_normalizing(data):
  data = np.array(data)

  bottom = data[data <= np.nanpercentile(data, 10, axis=0) ]
  bottom_avg = np.average(bottom)
  top = data[data >= np.nanpercentile(data, 90, axis=0) ]
  top_avg = np.average(top)
  normalized_data = (data - bottom_avg) / (top_avg - bottom_avg)  # Normalise with [min,max] -> [0,1]
  normalized_data = np.clip(normalized_data, 0.05, 1) # set to 0.05 all negative values, set to 1 all values greater than 1

  return normalized_data

# Variables

In [3]:
fs = 100
epoch_length = 10
mat_path = r"D:\dilon_data\MOTORWP4_dataset\sleep-edf_channel_test_mats"
files = np.ravel(os.listdir(mat_path))
eog_files = []
state_files = []

for file in files:
    if 'EOG' in file:
        eog_files.append(file)
    elif 'states' in file:
        state_files.append(file)

In [4]:
print(eog_files)
print(state_files)

[np.str_('S35_1_EOG.mat'), np.str_('S35_2_EOG.mat'), np.str_('S36_1_EOG.mat'), np.str_('S36_2_EOG.mat'), np.str_('S37_1_EOG.mat'), np.str_('S37_2_EOG.mat'), np.str_('S38_1_EOG.mat'), np.str_('S38_2_EOG.mat'), np.str_('S39_1_EOG.mat'), np.str_('S39_2_EOG.mat'), np.str_('S41_1_EOG.mat'), np.str_('S41_2_EOG.mat'), np.str_('S42_1_EOG.mat'), np.str_('S42_2_EOG.mat'), np.str_('S43_1_EOG.mat'), np.str_('S43_2_EOG.mat'), np.str_('S44_1_EOG.mat'), np.str_('S44_2_EOG.mat'), np.str_('S45_1_EOG.mat'), np.str_('S45_2_EOG.mat'), np.str_('S46_1_EOG.mat'), np.str_('S46_2_EOG.mat'), np.str_('S47_2_EOG.mat'), np.str_('S48_1_EOG.mat'), np.str_('S48_2_EOG.mat'), np.str_('S49_1_EOG.mat'), np.str_('S49_2_EOG.mat'), np.str_('S50_1_EOG.mat'), np.str_('S50_2_EOG.mat'), np.str_('S51_1_EOG.mat'), np.str_('S51_2_EOG.mat'), np.str_('S52_1_EOG.mat'), np.str_('S52_2_EOG.mat'), np.str_('S53_1_EOG.mat'), np.str_('S53_2_EOG.mat'), np.str_('S54_1_EOG.mat'), np.str_('S54_2_EOG.mat'), np.str_('S55_1_EOG.mat'), np.str_('S5

# Add EOG feature to existing hdf5 files

In [5]:
old_h5_file = r"D:\dilon_data\MOTORWP4_dataset\hdf5\motorwp4_fpz-cz_pz-oz.h5"
new_h5_file = r"D:\dilon_data\hdf5_collection\motorwp4_w_eog.h5"

In [6]:
subjects_eog_per_state = defaultdict(list)

for eog_file, state_file in zip(eog_files, state_files):
    eog_filepath = os.path.join(mat_path, eog_file)
    state_filepath = os.path.join(mat_path, state_file)

    # get subect
    subject = '_'.join(eog_file.split('_')[:2])
    #subject = subject[:-1] + '_' + subject[-1:]
    if 'SC' not in subject and 'ST' not in subject:
        fs = 250
    else:
        fs = 100
    # load EOG
    eog_data = loadmat(eog_filepath)
    eog_data = next(v for k, v in eog_data.items() if 'EOG' in k)
    raw_eog = np.ravel(eog_data)
    eog_data = raw_eog[:len(raw_eog) // (epoch_length * fs) * (epoch_length*fs)]
    eog_data = eog_data.reshape(-1, (epoch_length * fs))
    eog_data = eog_data.sum(axis=1)

    # # load states
    # sleep_scoring = loadmat(state_filepath)
    # states = sleep_scoring['states']
    # states = np.ravel(states)
    # sleep_scoring = np.array(states, dtype=int)
    # reshaped_scores = sleep_scoring[:len(sleep_scoring) // (epoch_length*fs) * (epoch_length*fs)].reshape(-1, epoch_length*fs)
    # majority_scores = mode(reshaped_scores, axis=1).mode.flatten()

    b, a = butter(4, [0.3, 35.0], btype='band', fs=fs)
    filtered_eog = filtfilt(b, a, eog_data, axis=-1)
    hilbert_eog = np.abs(hilbert(filtered_eog))
    hilbert_eog_norm = wei_normalizing(hilbert_eog)
    eog_smoothed = savgol_filter(hilbert_eog_norm, 11, polyorder=5)
    print(len(eog_smoothed))

    # for eog, state in zip(eog_smoothed, majority_scores):
    #     subjects_eog_per_state[state].append(eog)

    with h5py.File(old_h5_file, 'r') as old, h5py.File(new_h5_file, 'a') as new:
        if subject in list(old.keys()):
            # get old group object
            old_group = old[subject]

            if 'eog' not in old_group.attrs['Description features']:
                print('Adding EOG')
                # retrieve features and states from old 
                old_features = old_group['Features'][:]
                old_states = old_group['Mapped_scores'][:]

                # retrieve old description
                old_state_description = old_group.attrs['Description Mapped_scores']

                eog_smoothed = eog_smoothed.reshape(-1, 1)
                print(subject)
                print(len(old_features))
                print(len(eog_smoothed))
                new_features = np.column_stack((old_features, eog_smoothed[:len(old_features)]))

                try:
                    # create new group and add data
                    new_group = new.create_group(subject)
                except:
                    del new[subject]
                    new_group = new.create_group(subject)
                # features
                new_group.attrs['Description features'] = '[index_w_smoothed, index_r_smoothed, index_n_smoothed, index_1_smoothed, index_2_smoothed, index_3_smoothed, index_4_smoothed, noise_smoothed, theta_smoothed, delta_smoothed, aperiodic, dfa, mse, eog]'
                new_group.create_dataset('Features', data=new_features)
                # states
                new_group.attrs['Description Mapped_scores'] = old_state_description
                new_group.create_dataset('Mapped_scores', data=old_states)
                continue



    

2916
Adding EOG
S35_1
2916
2916
2769
Adding EOG
S35_2
2769
2769
3144
Adding EOG
S36_1
3144
3144
3414
Adding EOG
S36_2
3414
3414
2892
Adding EOG
S37_1
2892
2892
3048
Adding EOG
S37_2
3048
3048
3621
Adding EOG
S38_1
3621
3621
3177
Adding EOG
S38_2
3177
3177
3075
Adding EOG
S39_1
3075
3075
3054
Adding EOG
S39_2
3054
3054
3033
Adding EOG
S41_1
3033
3033
3105
Adding EOG
S41_2
3105
3105
3225
Adding EOG
S42_1
3225
3225
3003
Adding EOG
S42_2
3003
3003
3198
Adding EOG
S43_1
3198
3198
3057
Adding EOG
S43_2
3057
3057
3168
Adding EOG
S44_1
3168
3168
3198
Adding EOG
S44_2
3198
3198
3027
Adding EOG
S45_1
1910
3027
3453
Adding EOG
S45_2
3453
3453
3555
Adding EOG
S46_1
3555
3555
3147
Adding EOG
S46_2
3147
3147
3336
Adding EOG
S47_2
3336
3336
3147
Adding EOG
S48_1
3147
3147
3234
Adding EOG
S48_2
3234
3234
3192
Adding EOG
S49_1
3192
3192
3195
Adding EOG
S49_2
3195
3195
3360
Adding EOG
S50_1
3360
3360
3393
Adding EOG
S50_2
3393
3393
3135
Adding EOG
S51_1
3135
3135
2979
Adding EOG
S51_2
2979
2979
3276
Add

# Visualization

In [7]:
# states = [0, 1, 2, 3, 4]
# colors = ['royalblue', 'teal', 'purple', 'forestgreen', 'firebrick']

In [8]:
# means  = [np.mean(subjects_eog_per_state[s]) for s in states]
# sems   = [np.std(subjects_eog_per_state[s], ddof=1) / np.sqrt(len(subjects_eog_per_state[s])) for s in states]  # SEM

# plt.figure()
# plt.bar(states, means, yerr=sems, capsize=5, color=[colors[int(s)] for s in states], edgecolor='black', alpha=0.6)  # no colors specified
# plt.ylabel("Normalized hilbert transformed EOG value")
# plt.xticks(ticks=[0, 1, 2, 3, 4], labels=["Wake", "N1", "N2", "N3", "REM"])
# plt.title("Mean (+- SEM) EOG per Sleep State - Sleep-EDF Database Expanded")
# plt.tight_layout()
# plt.show()

In [9]:
# values = [subjects_eog_per_state[state] for state in states]

# plt.figure()
# box = plt.boxplot(values, labels=states, patch_artist=True)
# for patch, color in zip(box['boxes'], colors):
#     patch.set_facecolor(color)
#     patch.set_alpha(0.6)
#     patch.set_edgecolor('black')

# for median in box['medians']:
#     median.set_color("black")
#     median.set_linewidth(2)

# plt.ylabel("Normalized hilbert transformed EOG value")
# plt.xticks(ticks=[1, 2, 3, 4, 5], labels=["Wake", "N1", "N2", "N3", "REM"])
# plt.title("EOG per Sleep State - Sleep-EDF Database Expanded")
# plt.tight_layout()
# plt.show()
# plt.show()